<div align="center">

# Fine-Tuning Qwen 3.5-4B

🌐 **Language:** 🇬🇧 **English**

🤗 **Source:** [Jackrong](https://huggingface.co/Jackrong)

<br>

</div>

---

## Model & Dataset

- **Base Model**: Qwen3.5-4B (Unsloth 4-bit quantized)
- **Fine-tuning Method**: QLoRA (NF4 + Double Quantization)
- **Dataset**: [younissk/tool-calling-mix](https://huggingface.co/datasets/younissk/tool-calling-mix)
- **Samples**: ~68K (train + validation splits)
- **Tracking**: Weights & Biases


## Before You Start: Required API Keys 🔑

If you are new to Kaggle notebooks, please prepare **two API keys** before running the cells below, and store them in **Kaggle Secrets** first.

### 1. `WANDB_API_KEY`

- This key is used to log in to **Weights & Biases (W&B)**.
- In this notebook, it is used for experiment tracking, logging, and training visualization.
- Without it, the W&B login cell at the beginning will fail.

### 2. `HF_TOKEN`

- This key is used to log in to **Hugging Face**.
- In this notebook, it is mainly needed later if you want to **upload the trained model or GGUF files to Hugging Face Hub**.
- If you only want to train inside Kaggle and do not plan to upload artifacts, this key may not be needed immediately, but it is still recommended to prepare it in advance.

### How to store them in Kaggle Secrets

1. Open your Kaggle notebook.
2. Open the **Secrets** at the *Add-ons* on the topbar.
3. Add the following secret names exactly as written:
   - `WANDB_API_KEY`
   - `HF_TOKEN`
4. Paste the corresponding value for each key.
5. Save the secrets, then come back and run the notebook.

### Beginner Tip ✨

- Keep the secret **names** exactly the same as the code expects.
- Do not paste API keys directly into notebook code cells.
- Using Kaggle Secrets is the safer and cleaner way to manage credentials.


In [2]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()

try:
    wandb_key = user_secrets.get_secret("WANDB_API_KEY")
except:
    print("WANDB_API_KEY not found. W&B logging will be disabled.")
    wandb_key = None

if wandb_key:
    import wandb
    wandb.login(key=wandb_key)
    print("Logged in to W&B successfully!")
else:
    print("W&B login skipped - set WANDB_API_KEY in Kaggle Secrets to enable")

import os
output_directory = "/kaggle/working/"
os.makedirs(output_directory, exist_ok=True)
print(f"Checkpoints will be saved to: {output_directory}")

WANDB_API_KEY not found. W&B logging will be disabled.
W&B login skipped - set WANDB_API_KEY in Kaggle Secrets to enable
Checkpoints will be saved to: /kaggle/working/


In [3]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
!pip install transformers==5.3.0
!pip install --no-deps trl==0.22.2

In [4]:
# %%capture
# import os, re, sys

# def run_pip_install(cmd):
#     """Run pip install with error capture"""
#     result = os.popen(cmd + " 2>&1").read()
#     if result and ("error" in result.lower() or "exception" in result.lower() or "failed" in result.lower()):
#         print(f"pip warning: {result[:500]}", file=sys.stderr)
#     return result

# if "COLAB_" not in "".join(os.environ.keys()):
#     run_pip_install("!pip install unsloth")  # Do this in local & cloud setups
# else:
#     import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
#     xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
#     run_pip_install(f"!pip install sentencepiece protobuf \"datasets==4.3.0\" \"huggingface_hub>=0.34.0\" hf_transfer")
#     run_pip_install(f"!pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth")
# run_pip_install("!pip install transformers==5.3.0")
# run_pip_install("!pip install --no-deps trl==0.22.2")

In [5]:
# GPU Detection for Kaggle T4
print("=" * 50)
print("GPU Configuration")
print("=" * 50)
import torch
if torch.cuda.is_available():
    num_gpus = torch.cuda.device_count()
    print(f"Number of GPUs: {num_gpus}")
    for i in range(num_gpus):
        gpu_name = torch.cuda.get_device_name(i)
        vram_gb = torch.cuda.get_device_properties(i).total_memory / 1e9
        print(f"GPU {i}: {gpu_name} ({vram_gb:.1f}GB)")
    print("Using CUDA for training")
else:
    print("WARNING: No GPU detected - using CPU (will be very slow)")
print("=" * 50)

GPU Configuration
Number of GPUs: 2
GPU 0: Tesla T4 (15.6GB)
GPU 1: Tesla T4 (15.6GB)
Using CUDA for training


In [6]:
# Load Model and Tokenizer
import unsloth
from unsloth import FastLanguageModel
from transformers import BitsAndBytesConfig
import torch

# QLoRA Config - simplified (removed problematic env var)
bnb_config = BitsAndBytesConfig(
    load_in_4bit = True,
    bnb_4bit_quant_type = "nf4",
)

# Load model with explicit dtype for GPU
print("Loading model...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3-4B-Base",
    max_seq_length = 2048,
    load_in_4bit = True,
    load_in_8bit = False,
    full_finetuning = False,
    quantization_config = bnb_config,
    device_map = "auto",
    dtype = torch.float16,  # NEW: Ensures GPU usage
)
print("Model loaded successfully!")

# Verify GPU usage
print(f"Model device: {next(model.parameters()).device}")

print("Tokenizer loaded")

# Progress tracking
print(f"Tokenizer vocab size: {len(tokenizer)}")
print(f"Using device_map: auto")
print("Setup complete! Ready for LoRA configuration.")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Loading model...
==((====))==  Unsloth 2026.5.2: Fast Qwen3 patching. Transformers: 5.3.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/3.32G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/166 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/617 [00:00<?, ?B/s]

unsloth/qwen3-4b-base-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
Model loaded successfully!
Model device: cuda:0
Tokenizer loaded
Tokenizer vocab size: 151670
Using device_map: auto
Setup complete! Ready for LoRA configuration.


In [7]:
# ============================================================
# LoRA CONFIGURATION CONSTANTS
# ============================================================
# RANDOM_SEED=3407: Chosen for reproducibility. This specific seed
# provides consistent data shuffling and weight initialization.
# 3407 is a prime number, avoiding systematic patterns in splits.
#
# learning_rate=2e-4: Standard learning rate for LoRA fine-tuning.
# 2e-4 balances fast convergence with stability for 4-bit models.
# Lower rates risk slow convergence; higher rates risk divergence.
#
# LoRA rank (r)=64: Dimension of low-rank adaptation matrices.
# 64 provides good capacity for tool-calling task learning while
# keeping parameter overhead manageable (~33M params for 9B model).
# Higher r = more expressiveness but more compute/memory.
RANDOM_SEED = 3407
LEARNING_RATE = 2e-4
LORA_RANK = 64

model = FastLanguageModel.get_peft_model(
    model,
    r = LORA_RANK,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = LORA_RANK,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = RANDOM_SEED,
    use_rslora = True,
)

print("LoRA adapter configured with QLoRA optimization (RSLoRA enabled)")

Unsloth 2026.5.2 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


LoRA adapter configured with QLoRA optimization (RSLoRA enabled)


## Dataset: younissk/tool-calling-mix 📚

This dataset is designed for fine-tuning language models on tool use / function calling.

### Dataset Statistics

| Split | Samples |
|-------|--------|
| train | 60,600 |
| validation | 7,580 |
| test | 7,580 |
| **Total** | **75,760** |

### Data Composition

- **ToolBench Normalized**: 26.4%
- **xLAM60k**: 26.4%
- **OpenFunctions v1**: 15.2%
- **No-call (Dolly)**: 10.6%
- **No-call (WikiText)**: 10.6%
- **Synthetic Parallel**: 6.6%
- **Others**: 4.2%

### Key Features

- Average 1.34 tool calls per example
- Maximum 24 tool calls in a single example
- Includes non-tool calling examples to prevent catastrophic forgetting
- Tool definitions in JSON schema format


In [8]:
from datasets import load_dataset

print("Loading tool-calling-mix dataset (all splits)...")

dataset = load_dataset("younissk/tool-calling-mix")

print(f"\nDataset splits: {list(dataset.keys())}")
for split_name, split_data in dataset.items():
    print(f"  {split_name}: {len(split_data)} samples")

print(f"\nColumns: {dataset['train'].column_names}")
print(f"\nSample (first example):")
sample = dataset['train'][0]
print(f"  n_calls: {sample['n_calls']}")
print(f"  difficulty: {sample['difficulty']}")
print(f"  meta_source: {sample['meta_source']}")
print(f"  valid: {sample['valid']}")

Loading tool-calling-mix dataset (all splits)...


README.md: 0.00B [00:00, ?B/s]

raw/train.jsonl.gz:   0%|          | 0.00/47.8M [00:00<?, ?B/s]

raw/validation.jsonl.gz:   0%|          | 0.00/6.02M [00:00<?, ?B/s]

raw/test.jsonl.gz:   0%|          | 0.00/6.04M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/60648 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/7581 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7581 [00:00<?, ? examples/s]


Dataset splits: ['train', 'validation', 'test']
  train: 60648 samples
  validation: 7581 samples
  test: 7581 samples

Columns: ['tools_json', 'messages_json', 'target_json', 'meta_source', 'n_calls', 'difficulty', 'valid']

Sample (first example):
  n_calls: 2
  difficulty: parallel
  meta_source: toolbench_normalized
  valid: True


In [9]:
from datasets import load_dataset
dataset = load_dataset("younissk/tool-calling-mix")
# Print first example keys
print("Train keys:", dataset['train'].features.keys())
print("\nFirst example:")
example = dataset['train'][0]
for key in example.keys():
    print(f"  {key}: {type(example[key])}")
# Print messages structure if it exists
if 'messages' in example:
    print("\nMessages (first 2):")
    for msg in example['messages'][:2]:
        print(f"  Role: {msg.get('role')}")
        print(f"  Keys: {msg.keys()}")
        if 'tool_calls' in msg:
            print(f"  Tool calls: {msg['tool_calls']}")

Train keys: dict_keys(['tools_json', 'messages_json', 'target_json', 'meta_source', 'n_calls', 'difficulty', 'valid'])

First example:
  tools_json: <class 'str'>
  messages_json: <class 'str'>
  target_json: <class 'str'>
  meta_source: <class 'str'>
  n_calls: <class 'int'>
  difficulty: <class 'str'>
  valid: <class 'bool'>


In [10]:
# Dataset Preprocessing
import json
from datasets import concatenate_datasets

MAX_CONTEXT_LENGTH = 2048

def convert_to_training_format(example):
    try:
        messages = json.loads(example.get('messages_json', '[]'))
        conversations = []
        
        for msg in messages:
            role = msg.get('role', '')
            content = msg.get('content', '') or ''
            
            # Handle tool_calls (OpenAI format)
            tool_calls = msg.get('tool_calls', [])
            if tool_calls:
                # Structure: [{"type": "function", "function": {"name": "...", "arguments": "..."}}]
                for tc in tool_calls:
                    func = tc.get('function', {})
                    func_name = func.get('name', '')
                    func_args = func.get('arguments', '{}')
                    if isinstance(func_args, str):
                        func_args = func_args
                    else:
                        func_args = json.dumps(func_args)
                    
                    # Build tool call string with custom tags
                    content += f"\n<tool>\n<tool_name>\n{func_name}\n</tool_name>\n<tool_args>\n{func_args}\n</tool_args>\n</tool>"
            
            if role in ['user', 'assistant', 'system']:
                if role == 'system':
                    continue  # Skip duplicate system
                conversations.append({"role": role, "content": content})
        
        # Validation
        if len(conversations) < 2:
            return {"conversations": None}
        if conversations[-1]["role"] != "assistant":
            return {"conversations": None}
        
        return {"conversations": conversations}
    except Exception as e:
        print(f"Error: {e}, keys: {example.keys()}")  # DEBUG
        return {"conversations": None}

print("Combining train and validation splits...")
combined_dataset = concatenate_datasets([
    dataset['train'],
    dataset['validation']
])
print(f"Combined dataset: {len(combined_dataset)} samples")

print("Converting to training format...")
processed = combined_dataset.map(
    convert_to_training_format,
    remove_columns=combined_dataset.column_names,
    num_proc=4,
)
processed = processed.filter(lambda x: x['conversations'] is not None)
print(f"After filtering: {len(processed)} samples")

Combining train and validation splits...
Combined dataset: 68229 samples
Converting to training format...


Map (num_proc=4):   0%|          | 0/68229 [00:00<?, ? examples/s]

Filter:   0%|          | 0/68229 [00:00<?, ? examples/s]

After filtering: 38040 samples


In [11]:
# Apply Chat Template
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template="qwen3",
)

def apply_chat_template(example):
    text = tokenizer.apply_chat_template(
        example['conversations'],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

print("Applying chat template...")
dataset_final = processed.map(
    apply_chat_template,
    remove_columns=processed.column_names,
    num_proc=4,
)
print(f"Final dataset: {len(dataset_final)} samples")

# Filter by length
def filter_by_length(example):
    tokens = tokenizer(example['text'], add_special_tokens=False)
    return len(tokens['input_ids']) <= MAX_CONTEXT_LENGTH

before_count = len(dataset_final)
dataset_final = dataset_final.filter(filter_by_length, num_proc=4)
after_count = len(dataset_final)
print(f"Filtered: {before_count - after_count}/{before_count} samples removed ({100*(before_count-after_count)/before_count:.1f}%)")

Applying chat template...


Map (num_proc=4):   0%|          | 0/38040 [00:00<?, ? examples/s]

Final dataset: 38040 samples


Filter (num_proc=4):   0%|          | 0/38040 [00:00<?, ? examples/s]

Filtered: 212/38040 samples removed (0.6%)


In [12]:
# ============================================================
# TRAINING CONFIGURATION — TEST RUN (20 steps)
# ============================================================
# NOTE: This config is for quick testing only.
# - max_steps = 20: trains exactly 20 steps (~1 epoch on 320 samples)
# - For full training: set num_train_epochs = 1, remove max_steps
# Training Configuration
if len(dataset_final) == 0:
    raise ValueError("Dataset is empty after filtering.")

from trl import SFTTrainer, SFTConfig

wandb_project_name = "qwen-tool-calling-qlora"

if wandb_key:
    import os
    os.environ["WANDB_PROJECT"] = wandb_project_name

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset_final,
    eval_dataset = None,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 8,
        warmup_ratio = 0.04,
        max_steps = 20,
    # num_train_epochs = 2,  # test run
        learning_rate = LEARNING_RATE,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = RANDOM_SEED,
        save_steps = 100,
        save_total_limit = 2,
        save_strategy = "steps",
        report_to = "wandb" if wandb_key else "none",
        
        output_dir = output_directory,
        max_seq_length = MAX_CONTEXT_LENGTH,
        logging_dir = f"{output_directory}/logs",
    ),
)

print(f"Trainer configured:")
print(f"  - Batch size: 2 (per device)")
print(f"  - Gradient accumulation: 8")
print(f"  - Effective batch: 16")
print(f"  - Learning rate: {LEARNING_RATE}")
print(f"  - Epochs: 2")
print(f"  - Max sequence length: {MAX_CONTEXT_LENGTH}")
print(f"  - W&B project: {wandb_project_name}")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/37828 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.
Trainer configured:
  - Batch size: 2 (per device)
  - Gradient accumulation: 8
  - Effective batch: 16
  - Learning rate: 0.0002
  - Epochs: 2
  - Max sequence length: 2048
  - W&B project: qwen-tool-calling-qlora


In [13]:
# Train on responses only
from unsloth.chat_templates import train_on_responses_only

INSTRUCTION_MARKER = '<|im_start|>user\n'
RESPONSE_MARKER = '<|im_start|>assistant\n'

trainer = train_on_responses_only(
    trainer,
    instruction_part = INSTRUCTION_MARKER,
    response_part = RESPONSE_MARKER,
)

print("Training configured to learn only from assistant responses")

Map (num_proc=8):   0%|          | 0/37828 [00:00<?, ? examples/s]

Filter (num_proc=8):   0%|          | 0/37828 [00:00<?, ? examples/s]

Training configured to learn only from assistant responses


## Training — Test Run

This cell runs the actual fine-tuning for **20 steps only** (test run).

- **Why 20 steps?** Quick validation that the pipeline works — labels are not -100, loss is non-zero.
- **For full training:** Run this cell again with `num_train_epochs = 1` after confirming the test passes.

In [14]:
print("=" * 50)
print("Starting training...")
print("=" * 50)

trainer.train()

print("=" * 50)
print("Training complete!")
print("=" * 50)

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Starting training...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 2
   \\   /|    Num examples = 37,828 | Num Epochs = 1 | Total steps = 20
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 132,120,576 of 4,154,588,672 (3.18% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
1,1.284890
2,1.584267
3,1.059021
4,0.976995
5,1.332739
6,0.976969
7,0.871997
8,0.877951
9,1.013392
10,1.199107


Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/checkpoint-20/tokenizer_config.json.


Training complete!


## Push LoRA Adapter to HuggingFace Hub

In [17]:
# Push LoRA Adapter to HuggingFace Hub
from huggingface_hub import whoami

hf_token = user_secrets.get_secret("HF_TOKEN")
user_info = whoami(token=hf_token)
username = user_info['name']
lora_repo_id = f"{username}/qwen3-4b-tool-calling-lora"

print(f"Authenticated as: {username}")
print(f"Uploading LoRA adapter to {lora_repo_id}...")

model.push_to_hub(lora_repo_id, token=hf_token)
tokenizer.push_to_hub(lora_repo_id, token=hf_token)

print(f"LoRA pushed: https://huggingface.co/{lora_repo_id}")

Authenticated as: jonlimanza
Uploading LoRA adapter to jonlimanza/qwen3-4b-tool-calling-lora...


README.md:   0%|          | 0.00/568 [00:00<?, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Saved model to https://huggingface.co/jonlimanza/qwen3-4b-tool-calling-lora


Unsloth: Restored added_tokens_decoder metadata in /tmp/tmprn050dms/tokenizer_config.json.


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

LoRA pushed: https://huggingface.co/jonlimanza/qwen3-4b-tool-calling-lora


## Save Model

Export trained artifacts. Two options:

1. **LoRA Adapter** — lightweight (~100MB), already pushed to HF Hub (see above)
2. **Merged 16-bit** — full model (~8GB), may fail on Kaggle disk limit (~20GB)

## Note Before Exporting GGUF Models ⚠️

- Please note that the temporary storage available to a Kaggle notebook is effectively capped at around **92 GB** in practice, even though the interface may show **50 GB**.
- Because of this limit, in some cases you may **not be able to export every GGUF size directly from Kaggle**.
- A practical workaround is to first upload the **16-bit model**, then use the **Colab quantization notebook** I published to quantize and release the full set of model sizes in Colab.
- Colab usually provides **more temporary storage**, so it is better suited for exporting all GGUF variants. 🚀


**Note:** Because the pipeline was tested on Kaggle (limited disk space ~20GB), the merged model save and GGUF export cells below may fail. GGUF export is deferred to a local machine or Colab with more disk space.

### ⚠️ Merged 16-bit Model Save (May Fail on Kaggle)

This step merges LoRA adapters with the FP16 base model and saves to disk (~8GB).

**Known issue:** Kaggle disk limit (~20GB) may cause this to fail with "no disk space left".

**If it fails:** Skip this cell. The LoRA adapter (already pushed to HF Hub) is sufficient for most use cases. Alternatively, export GGUF on a local machine with more storage.

**If it succeeds:** Proceed to GGUF export cells below.

In [18]:
# Save Merged 16bit Model
merged_save_path = f"{output_directory}qwen_tool_calling_merged"
model.save_pretrained_merged(
    merged_save_path,
    tokenizer,
    save_method = "merged_16bit",
)
print(f"Merged 16bit model saved to: {merged_save_path}")

config.json:   0%|          | 0.00/752 [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/qwen_tool_calling_merged/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Checking cache directory for required files...
Cache check failed: model-00001-of-00002.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


...

model-00002-of-00002.safetensors:   0%|          | 0.00/3.08G [00:00<?, ?B/s]


Unsloth: Preparing safetensor model files: 100%|██████████| 2/2 [00:33<00:00, 16.73s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [00:55<00:00, 27.85s/it]


Unsloth: Merge process complete. Saved to `/kaggle/working/qwen_tool_calling_merged`
Merged 16bit model saved to: /kaggle/working/qwen_tool_calling_merged


### Try Direct GGUF Export (no llama.cpp needed)

Attempts to export the model directly to GGUF Q4_K_M format using Unsloth's built-in method.

**Note:** This step requires significant temporary disk space (~8GB for F16 interim + ~2.4GB for Q4_K_M output). If disk space is insufficient, this cell will fail and the fallback (see below) will trigger automatically.

In [20]:
# Try Direct GGUF Export (no llama.cpp needed)
gguf_path = f"{output_directory}qwen_tool_calling_q4km"
try:
    model.save_pretrained_gguf(
        gguf_path,
        tokenizer,
        quantization_method = "q4_k_m",
    )
    print(f"GGUF Q4_K_M saved to: {gguf_path}")
    GGUF_SUCCESS = True
except Exception as e:
    print(f"Direct GGUF export failed: {e}")
    print("Falling back to llama.cpp conversion...")
    GGUF_SUCCESS = False

Unsloth: Merging model weights to 16-bit format...
Direct GGUF export failed: Failed to save/merge model: Unsloth: Failed saving locally - no disk space left. Uploading can work luckily! Use .push_to_hub instead.
Falling back to llama.cpp conversion...


### llama.cpp Fallback (if direct GGUF failed)

If the direct GGUF export above failed due to disk space, this cell clones llama.cpp and converts the merged model manually.

**Note:** This fallback also requires disk space for the intermediate conversion. If disk space is still insufficient, skip cells 23-24 entirely. The LoRA adapter on HuggingFace Hub is the primary available artifact.

In [ ]:
# llama.cpp Fallback (if direct GGUF failed)
if not GGUF_SUCCESS:
    print("Cloning llama.cpp...")
    !git clone --depth 1 https://github.com/ggerganov/llama.cpp.git
    print("Converting to GGUF Q4_K_M...")
    !python llama.cpp/convert_hf_to_gguf.py qwen_tool_calling_merged \
        --outfile qwen_tool_calling_q4km.gguf --outtype q4_k_m \
        --split-max-size 2G
    print(f"GGUF Q4_K_M saved via llama.cpp")
else:
    print("Skipping llama.cpp - direct export succeeded")

### Push GGUF Q4_K_M to HuggingFace Hub

Uploads the GGUF Q4_K_M file to HuggingFace Hub.

**Note:** Upload size is ~2.4GB. If the GGUF file does not exist (due to failed export above), this cell will also fail. In that case, the LoRA adapter on HuggingFace Hub is the primary available artifact.

In [ ]:
# Push GGUF Q4_K_M to HuggingFace Hub
from huggingface_hub import HfApi, whoami

hf_token = user_secrets.get_secret("HF_TOKEN")
user_info = whoami(token=hf_token)
username = user_info['name']
gguf_repo_id = f"{username}/qwen3-4b-tool-calling-q4km"

print(f"Authenticated as: {username}")
gguf_file = f"{gguf_path}/qwen_tool_calling_q4km.gguf" if GGUF_SUCCESS else f"{output_directory}qwen_tool_calling_q4km.gguf"
print(f"Uploading GGUF Q4_K_M to {gguf_repo_id}...")

api = HfApi()
api.upload_file(
    path_or_fileobj = gguf_file,
    path_in_repo = "qwen_tool_calling_q4km.gguf",
    repo_id = gguf_repo_id,
    repo_type = "model",
    token = hf_token,
)
print(f"GGUF pushed: https://huggingface.co/{gguf_repo_id}")